# Fine-tune **Qwen2.5-VL-7B v2** on the TTB-live stratified 18K (Colab + Drive)

LoRA fine-tune mirroring `notebooks/sft_qwen2_5_vl.ipynb` (v1), with these changes:

- **Training data: 18K TTB-live composites** (vs v1's 1377 COLA Cloud singles) — sourced
  by `test/eval/scrape_ttb_registry.py` on the Mac, lives at `MyDrive/ttb-scrape/images/`.
- **Stratified by beverage**: ~6K spirits / 6K wine / 8K beer-malt (TTB classTypeCode buckets).
- **Targets are fields-only** (Brand name, Class & type, Bottler name/address, Country of origin)
  sourced verbatim from TTB Form 5100.31. Alcohol content / Net contents / Government warning
  are NOT carried on the form; teaching them from model output would just bake in the
  upstream model's mistakes.
- **2K held-out test set** at `MyDrive/ttb-scrape/holdout_ttbids.txt` — the trainer never sees
  it; the comparison notebook scores v1, v2, and Haiku against it.
- **Adapter saves to `MyDrive/ttb_sft/qwen2_5_vl_7b_v2/`** so v1 and v2 coexist for the head-to-head.
- **1 epoch** (vs v1's 2): 18K × 1 ≈ 6× more unique signal than v1's 1377 × 2;
  ~3-4 hr on A100, ~6-8 hr on T4 with reduced batch.

## How to run
1. `Runtime → Change runtime type → GPU` (A100 preferred; T4 works with smaller `MAX_PIXELS`)
2. `Runtime → Run all` — installs deps, restarts kernel
3. Wait for "session crashed" banner, then `Runtime → Run all` AGAIN
4. Training proceeds end-to-end; adapter saves to Drive automatically


## 0. Install dependencies

In [ ]:
# Pinned to match notebooks/sft_qwen2_5_vl.ipynb so v1 and v2 train +
# eval under the same stack. First Run all installs, kernel auto-restarts,
# second Run all uses cached wheels.
import importlib.util, subprocess, sys, os

def _have(mod):  return importlib.util.find_spec(mod) is not None
def _pinned(mod, want):
    try: return __import__(mod).__version__ == want
    except Exception: return False
def _numpy_ok():
    try:
        import numpy
        p = numpy.__version__.split('.')
        return int(p[0]) == 2 and int(p[1]) < 2
    except Exception: return False
def _torchao_ok():
    try:
        import torchao
        maj, mi = [int(x) for x in torchao.__version__.split('.')[:2]]
        return (maj, mi) >= (0, 16)
    except Exception: return False

_required = ['peft', 'trl', 'accelerate', 'PIL', 'qwen_vl_utils', 'datasets']
_ready = (
    _numpy_ok()
    and _pinned('transformers', '4.51.3')
    and _torchao_ok()
    and all(_have(m) for m in _required)
)
if _ready:
    import numpy, transformers, torchao
    print(f'✓ Stack ready — numpy {numpy.__version__}, transformers {transformers.__version__}, torchao {torchao.__version__}')
else:
    print('⏳ Installing deps (~3 min). Kernel will auto-restart at end.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
        'transformers==4.51.3', 'accelerate>=0.30',
        'peft>=0.13', 'torchao>=0.16', 'bitsandbytes>=0.44', 'trl>=0.12',
        'qwen-vl-utils', 'protobuf>=5.27', 'pillow', 'tqdm',
        'datasets>=2.20', 'anthropic>=0.40'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        '--force-reinstall', '--no-deps', 'numpy>=2.0,<2.2'])
    print('\n' + '=' * 60)
    print('✅ Install complete. Auto-restarting kernel.')
    print('   When the "session crashed" banner appears, do Runtime → Run all AGAIN.')
    print('=' * 60)
    os.kill(os.getpid(), 9)


## 1. Validate imports (fails fast on ABI mismatch)

In [ ]:
import importlib, traceback
import numpy
if numpy.__version__.startswith('1.'):
    raise RuntimeError(f'numpy is {numpy.__version__}; restart kernel + Run all again.')
print(f'  ✓ numpy {numpy.__version__}')
_required = ['torch', 'transformers', 'accelerate', 'peft', 'trl',
             'datasets', 'PIL', 'tqdm', 'torchao', 'qwen_vl_utils']
for mod in _required:
    m = importlib.import_module(mod)
    print(f'  ✓ {mod:<16} {getattr(m, "__version__", "?")}')
print('\n✅ All imports clean.')


## 2. Mount Drive + set up paths

In [ ]:
# Drive holds the TTB scrape (~9 GB) and the trained adapters. We mount
# the same Drive that the Mac scraper writes to (My Drive/ttb-scrape/).
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT  = Path('/content/drive/MyDrive')
SCRAPE_DIR  = DRIVE_ROOT / 'ttb-scrape'                  # set by the Mac scraper
IMAGE_DIR   = SCRAPE_DIR / 'images'                       # ~20K composites
JSONL_DIR   = SCRAPE_DIR / 'qwen_sft'                     # staged by overnight_pipeline.sh
CSV_PATH    = SCRAPE_DIR / 'ttb_live.csv'                 # form data
HOLDOUT_TXT = SCRAPE_DIR / 'holdout_ttbids.txt'           # 2K eval set, blind to trainer

for p in (SCRAPE_DIR, IMAGE_DIR, JSONL_DIR):
    print(f'  {"✓" if p.exists() else "✗"} {p}')
print(f'  composites: {len(list(IMAGE_DIR.glob("*.jpg"))) if IMAGE_DIR.exists() else 0}')


In [ ]:
# v2 adapter destination — distinct from v1 so they coexist
MODEL_TAG = 'qwen2_5_vl_7b_v2'
BASE_MODEL = 'Qwen/Qwen2.5-VL-7B-Instruct'
WORK_DIR = DRIVE_ROOT / 'ttb_sft' / MODEL_TAG
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f'work dir: {WORK_DIR}')
print(f'v1 adapter (for comparison eval later): {DRIVE_ROOT / "ttb_sft" / "qwen2_5_vl_7b"}')


## 3. Load training JSONL from Drive

Built by `test/eval/build_qwen_jsonl.py` on the Mac and staged to Drive by
`scripts/overnight_pipeline.sh`. The JSONL excludes the 2K holdout.


In [ ]:
import json
def _load_jsonl(path):
    return [json.loads(line) for line in open(path) if line.strip()]

train_raw = _load_jsonl(JSONL_DIR / 'train.jsonl')
val_raw   = _load_jsonl(JSONL_DIR / 'val.jsonl')
print(f'train: {len(train_raw)}   val: {len(val_raw)}')

# ── LEAKAGE GUARD — hard-stop if any train/val ttb_id is in the holdout ──
# This is defense-in-depth: build_qwen_jsonl.py already excludes the
# holdout when writing the JSONL, but a stale JSONL + fresh holdout (or
# vice-versa) would silently contaminate the training set without this
# assertion.
holdout_ids = {line.strip() for line in HOLDOUT_TXT.read_text().splitlines() if line.strip()}
print(f'holdout list size: {len(holdout_ids)}')
train_ids = {r['ttb_id'] for r in train_raw}
val_ids   = {r['ttb_id'] for r in val_raw}
leak_train = train_ids & holdout_ids
leak_val   = val_ids   & holdout_ids
assert not leak_train, f'LEAKAGE: {len(leak_train)} train ttb_ids appear in holdout. First 5: {list(leak_train)[:5]}'
assert not leak_val,   f'LEAKAGE: {len(leak_val)} val ttb_ids appear in holdout. First 5: {list(leak_val)[:5]}'
assert not (train_ids & val_ids), f'LEAKAGE: train ∩ val is non-empty ({len(train_ids & val_ids)} ttb_ids)'
print(f'✓ train ∩ holdout = ∅   val ∩ holdout = ∅   train ∩ val = ∅')

# Rewrite image_path from Mac-relative ('test/eval/data/ttb_live/images/<f>.jpg')
# to Drive-absolute. Strip prefix, prepend IMAGE_DIR.
def _fix_paths(rows):
    for r in rows:
        r['image_path'] = str(IMAGE_DIR / Path(r['image_path']).name)
    return rows

train = _fix_paths(train_raw)
val   = _fix_paths(val_raw)
# Drop any rows whose composite is missing on Drive (in case scrape was partial)
train = [r for r in train if Path(r['image_path']).exists()]
val   = [r for r in val   if Path(r['image_path']).exists()]
print(f'after image-presence filter:  train={len(train)}  val={len(val)}')

# Beverage distribution (sanity check stratification held)
from collections import Counter
print('train beverage mix:', Counter(r['beverage_type'] for r in train))


## 4. System prompt + image preprocessing

In [ ]:
SYSTEM_PROMPT = (
    "You are a careful transcription assistant for U.S. TTB alcohol label review. "
    "You will see a single image that may contain MULTIPLE label panels stacked "
    "vertically (front + back + neck + strip). READ what is printed across ALL panels "
    "and RETURN ONLY a JSON object matching the schema below. Do NOT decide compliance.\n\n"
    "CRITICAL — use ONLY these EXACT field names in the fields object (case-sensitive, "
    "preserve spaces and punctuation):\n"
    "  - 'Brand name'\n"
    "  - 'Class & type'\n"
    "  - 'Bottler name/address'\n"
    "  - 'Country of origin'  (imports only; omit for U.S. labels)\n\n"
    "Do NOT invent variants ('brand_name', 'Volume', 'Bottled_by', 'product_type', "
    "'location', etc). If a field is not clearly visible across the panels, OMIT it "
    "from the fields object entirely. Never guess; never substitute deposit codes "
    "(e.g. 'CA CRV'), NOM IDs, or barcodes.\n\n"
    "Schema: {fields: {<field name>: {value, confidence}}}"
)


In [ ]:
# Image preprocessing is delegated to qwen_vl_utils.process_vision_info,
# the canonical Qwen2.5-VL pattern. The critical detail: both
# apply_chat_template AND process_vision_info MUST see the same
# max_pixels — otherwise the chat template counts placeholders for
# one image size while the processor produces features for another.
# We pass max_pixels at three places (processor init, in-message, and
# in the smart_resize default) to lock alignment.
#
# Using v1's tested 1M cap (vs my earlier 1.5M) gives ~1296 visual
# tokens with comfortable headroom under max_length=4096 even for
# tall multi-panel composites. v1 trained successfully at this cap.
MAX_PIXELS_TRAIN = 1024 * 1024


## 5. Load Qwen2.5-VL-7B base + LoRA rank 16

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from peft import LoraConfig, get_peft_model

print(f'Loading {BASE_MODEL} BF16...')
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='sdpa',
)
# CRITICAL: pass max_pixels at processor init so apply_chat_template's
# internal smart_resize matches the cap we use in process_vision_info.
# Mismatch here is what caused the `tokens != features` ValueError.
tokenizer = AutoProcessor.from_pretrained(
    BASE_MODEL,
    max_pixels=MAX_PIXELS_TRAIN,
    min_pixels=256 * 28 * 28,   # ~200K — Qwen default; prevents over-shrink on small panels
)

# Freeze vision tower (matches v1)
for n, p in model.named_parameters():
    if 'visual' in n:
        p.requires_grad = False

# LoRA rank 16 / alpha 32 / dropout 0.05 — slight nudge from v1's rank 16/alpha 16/dropout 0
# to give the wider 18K dataset more capacity while regularizing against overfit.
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()
model.gradient_checkpointing_enable()
print('✓ gradient_checkpointing_enable — disable manually if you have A100-80GB headroom')


## 6. HF Dataset + per-example collator (Qwen-VL needs batch_size=1)

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_list([
    {'image_path': ex['image_path'], 'beverage_type': ex['beverage_type'],
     'target_json': json.dumps(ex['target'])}
    for ex in train
])
val_ds = Dataset.from_list([
    {'image_path': ex['image_path'], 'beverage_type': ex['beverage_type'],
     'target_json': json.dumps(ex['target'])}
    for ex in val
])
print(f'train_ds: {len(train_ds)}   val_ds: {len(val_ds)}')

from qwen_vl_utils import process_vision_info

def _collate(batch):
    assert len(batch) == 1, 'Qwen2.5-VL collator expects batch_size=1'
    ex = batch[0]
    # Pass the image PATH (not a PIL.Image) so process_vision_info can
    # load + resize it through the SAME smart_resize function the
    # processor uses internally. This is what keeps placeholder count
    # and feature count synchronized.
    msgs = [
        {'role': 'system',    'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user',      'content': [
            {'type': 'image', 'image': ex['image_path'], 'max_pixels': MAX_PIXELS_TRAIN},
            {'type': 'text',  'text': f"Beverage type: {ex['beverage_type']}. Extract per the schema."},
        ]},
        {'role': 'assistant', 'content': [{'type': 'text', 'text': ex['target_json']}]},
    ]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    image_inputs, _video = process_vision_info(msgs)
    # max_length=4096 gives headroom: 1296 image tokens + ~300 system + ~30 user text
    # + ~250 target + ~30 chat template overhead ≈ 1900 — well under 4096.
    # truncation_side='left' (defensive): if anything ever overflows, drop early
    # tokens before image, never mid-image-block (which causes tokens!=features).
    enc = tokenizer(text=[text], images=image_inputs, padding=True, return_tensors='pt',
                    truncation=True, max_length=4096)
    labels = enc['input_ids'].clone()
    pad_id = tokenizer.tokenizer.pad_token_id
    if pad_id is not None:
        labels[labels == pad_id] = -100
    enc['labels'] = labels
    return enc


## 7. Smoke test — 1 step on 4 samples

In [ ]:
from trl import SFTTrainer, SFTConfig

_smoke = train_ds.select(range(min(4, len(train_ds))))
SFTTrainer(
    model=model,
    train_dataset=_smoke,
    data_collator=_collate,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        max_steps=1, warmup_steps=0,
        learning_rate=1e-4,
        bf16=True, fp16=False,
        logging_steps=1, output_dir='/tmp/smoke',
        report_to='none', remove_unused_columns=False,
        dataset_kwargs={'skip_prepare_dataset': True},
    ),
).train()
print('\n✅ SMOKE TEST PASSED — full training is safe to start.')


## 8. Full training run (1 epoch over 18K, BF16)

| Hardware | Wall time | Fits Colab? |
|---|---|---|
| A100-80GB (Pro+) | ~12-14 hr | ✅ comfortable under 24 hr cap |
| A100-40GB (Pro+) | ~17-20 hr | ⚠️  tight under 24 hr cap |
| L4 (Pro) | ~25-30 hr | ❌ needs 2-3 resumes |
| T4 (free) | ~50+ hr | ❌ impractical |

**Session-disconnect insurance.** This cell:
- Checkpoints to Drive every **500 steps** (~30 min on A100). Worst-case loss = 30 min of training.
- Uses `resume_from_checkpoint=True` — if you re-run this cell after a Colab disconnect,
  `Trainer` auto-resumes from the latest checkpoint in `MyDrive/ttb_sft/qwen2_5_vl_7b_v2/checkpoints/`.
  Tracks: epoch, step, optimizer state, LR scheduler position, RNG state.
- Keeps the last 3 checkpoints (older ones auto-deleted) so Drive doesn't bloat —
  ~3 × 500 MB = 1.5 GB.
- **For Pro+ background execution:** in the Colab menu, Runtime → Manage sessions →
  enable "Background execution" so the tab can close without killing the run.

If a Colab session dies and you re-run this cell, you'll see logs like
`Continuing training from checkpoint, will skip to global_step N` — that's the resume.


In [ ]:
from trl import SFTTrainer, SFTConfig

CKPT_DIR = WORK_DIR / 'checkpoints'
has_ckpt = CKPT_DIR.exists() and any(CKPT_DIR.glob('checkpoint-*'))
if has_ckpt:
    latest = max(CKPT_DIR.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]))
    print(f'↻ Found prior checkpoint at {latest.name} — Trainer will resume from there.')
else:
    print('▶ No prior checkpoint — starting fresh.')

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=_collate,
    args=SFTConfig(
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,     # MUST match train — collator asserts batch=1
        gradient_accumulation_steps=4,
        warmup_ratio=0.03,
        num_train_epochs=1,
        learning_rate=1e-4,
        bf16=True, fp16=False,
        lr_scheduler_type='cosine',
        weight_decay=0.01,
        optim='adamw_8bit',
        logging_steps=20,
        eval_strategy='steps', eval_steps=1000,
        save_strategy='steps', save_steps=500, save_total_limit=3,
        seed=42,
        output_dir=str(CKPT_DIR),
        report_to='none',
        remove_unused_columns=False,
        dataset_kwargs={'skip_prepare_dataset': True},
    ),
)
trainer.train(resume_from_checkpoint=has_ckpt)


## 9. Save adapter to Drive (v2 path — coexists with v1)

In [ ]:
import shutil
ADAPTER_DIR = WORK_DIR / 'adapter'
ADAPTER_DIR.mkdir(exist_ok=True)
model.save_pretrained(str(ADAPTER_DIR))
try: tokenizer.save_pretrained(str(ADAPTER_DIR))
except Exception as e: print(f'(skip processor save: {e})')

_safetensors = ADAPTER_DIR / 'adapter_model.safetensors'
_sz = _safetensors.stat().st_size if _safetensors.exists() else 0
assert _sz > 1_000_000, f'adapter_model.safetensors is {_sz} bytes — LoRA weights not saved.'
print(f'✓ adapter: {_safetensors} ({_sz/1024**2:.1f} MB)')

manifest = {
    'model_tag':         MODEL_TAG,
    'base_model':        BASE_MODEL,
    'trained_on':        f'TTB-live stratified N={len(train)} (fields-only targets)',
    'lora_rank':         16,
    'lora_alpha':        32,
    'epochs':            1,
    'lr':                1e-4,
    'n_train_examples':  len(train),
    'n_val_examples':    len(val),
    'training_dtype':    'bf16',
    'schema':            'fields-only (no government_warning, no image_quality)',
}
(ADAPTER_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print(f'manifest: {ADAPTER_DIR / "manifest.json"}')

# Bundle for browser download (optional — adapter is already on Drive)
import os
zip_path = shutil.make_archive(str(WORK_DIR / f'ttb_sft_{MODEL_TAG}'), 'zip',
                                root_dir=str(WORK_DIR), base_dir='adapter')
size_mb = os.path.getsize(zip_path) / 1024**2
print(f'\nzip bundle: {zip_path}  ({size_mb:.1f} MB)')
print(f'(adapter is already on Drive at {ADAPTER_DIR}; zip is just a convenience)')


## 10. Next: run the comparison notebook

Open `notebooks/eval_v1_vs_v2_drive.ipynb` in Colab — it loads both v1 and v2 adapters
from Drive, runs them on the 2K holdout, calls Haiku via the Anthropic API for a third
baseline, and writes `MyDrive/ttb_sft/eval/v1_vs_v2_comparison.md`.
